# Tennis Data

Before proceeding with the main tasks, make sure to complete the initial setting by running the following code.

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import itertools
from matplotlib import rc
from package import algorithm
from joblib import Parallel, delayed

rc('text', usetex=True)
rc('font', family='serif')
sz = 36

pd.set_option('future.no_silent_downcasting', True)
current_path = os.getcwd()
data_path = current_path + '\\' + 'data' + '\\' + 'tennis' + '\\' 
image_path = current_path + '\\' + 'image'+ '\\' + 'tennis'+ '\\'

/Users/cx/Dropbox/rank data analysis/Ranking with dynamic covariates/github code/package/basicfun.py:126: SyntaxWarning: invalid escape sequence '\w'
  label = [f'{LLabel} $\\mathbf{{\widehat{{u}}}}$ error',
/Users/cx/Dropbox/rank data analysis/Ranking with dynamic covariates/github code/package/basicfun.py:127: SyntaxWarning: invalid escape sequence '\w'
  f'{LLabel} $\\mathbf{{\widehat{{v}}}}$ error']


## Data Preprocessing

If the file 'tennis(preprocessed).csv' is already present within the folder 'data/tennis', then you have the option to skip this particular module.

In [2]:
covariate = ['winner_age', 'loser_age']
df = combined_df = pd.DataFrame()
for file in os.listdir(data_path):
    if 'atp_matches' in file:
        tem = pd.read_csv(data_path+file,)
        tem = tem.drop(tem[tem['score'] == 'W/O'].index)
        tem = tem.dropna(subset=covariate)
        df = pd.concat([df, tem], ignore_index=True)
    else:
        pass

FileNotFoundError: [Errno 2] No such file or directory: '/Users/cx/Dropbox/rank data analysis/Ranking with dynamic covariates/github code\\data\\tennis\\'

Select suitable matches and players <br>
<span style="font-size: smaller;">
We remove players whose competition times are less than 10 or who have never won.
</span> 

In [ ]:
def delete_players(df,d_min):
    players = df[['winner_name','loser_name']]
    counts = pd.Series(players.values.ravel()).value_counts()
    ### d >= d_min
    remain_players = counts[counts>=d_min].index
    ### winner
    remain_players = remain_players[remain_players.isin(df['winner_name'])].tolist()
    n = len(remain_players)
    return remain_players,n
def delete_matches(df,remain_players):
    df = df[df[['winner_name','loser_name']].isin(remain_players).all(axis=1)]
    N = len(df)
    return df,N
def select(df,d_min=10):
    players = df[['winner_name','loser_name']]
    counts = pd.Series(players.values.ravel()).value_counts()
    all_players = counts.index.tolist()
    nn = len(all_players)
    NN = len(df)
    while True:
        print('-'*10)
        print(f'The number of matches: {NN}')
        print(f'The number of players: {nn}')
        remain_players,n = delete_players(df,d_min)
        df,N = delete_matches(df,remain_players)
        if NN == N and nn == n:
            break
        else:
            NN = N
            nn = n
    return df,n,N    
df,n,N = select(df,10)

In [ ]:
df.to_csv(data_path+'tennis(preprocessed).csv',index=None)

## Data Analysis

Load the preprocessed tennis data that is located in the 'tennis(preprocessed).csv' file.

In [ ]:
df = pd.read_csv(data_path+'tennis(preprocessed).csv',low_memory=False)

# players ID
players = df[['winner_name','loser_name']]
counts = pd.Series(players.values.ravel()).value_counts()
playerID = {value: index for index, value in enumerate(counts.index.tolist())}
n = len(playerID)
# matches
name_columns = ['winner_name','loser_name']
Matches = df[name_columns].replace(playerID)
T = np.array(Matches).tolist()
N = len(T)
# age
age_columns = ['winner_age','loser_age']
Age = np.array(df[age_columns])


Kernel functions

In [ ]:
A = [25,30,35]
Lamb = [0.01, 0.03]
parameters = [(a,lamb) for a in A for lamb in Lamb]
gauss_kernel = lambda x,a,lamb: np.exp(-lamb*(x-a)**2)
gauss_kernel_set = lambda x: np.array([gauss_kernel(x,a,lamb) for (a,lamb) in parameters])
cov = [gauss_kernel_set(age).T for age in Age]

### Model selection

BIC criteria

In [ ]:
# BIC
def get_BIC(T,cov,s,N,n):
    X = [x[:,s] for x in cov]
    d = len(s)
    u_plusDC,v_plusDC = algorithm.AM(T,X,n,d,E=1e-3,Eu=1e-8,Ev=1e-12,type = 'pair')
    likelihood = algorithm.multi_likelihood(T,X,u_plusDC,v_plusDC)
    BIC = (n-1+d) * np.log(N) - 2*likelihood*N
    return BIC

# get_subset
get_subset = lambda n: [subset for i in range(n + 1)
                         for subset in itertools.combinations(list(range(n)), i)]
subset = get_subset(6)

# run
res = Parallel(n_jobs=7)(delayed(get_BIC)(T,cov,subset[i],N,n) for i in range(len(subset)))
s = subset[np.argmin(res)]
print('selected subset:', s)



### Fit Model

In [ ]:
s = (0, 1, 2, 4)
d = len(s)
print('selected subset:', s)
X = [x[:,s] for x in cov]

PlusDC

In [ ]:
u_plusDC,v_plusDC = algorithm.AM(T,X,n,d,
                E=1e-4/N,Eu=1e-8,Ev=1e-12,
                I=52,type = 'pair',detail=True)

likelihood_plusDC = algorithm.pair_likelihood(T,X,u_plusDC,v_plusDC)
print(likelihood_plusDC)

Ranking (PlusDC)

In [ ]:
plusDC_top10 = np.argsort(u_plusDC)[-20:][::-1]
u_t10_plusDC = u_plusDC[plusDC_top10]
top_player = []

for i,index in enumerate(plusDC_top10):
    player_name = [key for key, value in playerID.items() if value == index][0]
    top_player.append(player_name)
    print(f'top-{i+1}: player: {player_name}, score: {u_t10_plusDC[i]}')

Ranking (BT)

In [ ]:
"""u_BT,v_BT = algorithm.AM(T,X,n,
                E=1e-4/N,Eu=1e-8,Ev=1e-12,P=True,
                I=52,type = 'pair',detail=True)"""
v=np.array([0]*d)
KK = np.array([k[0] - k[1] for k in X])
u_BT = algorithm.pair_fixv(T,KK,v,n,E = 1e-8,I=52)
likelihood_BT = algorithm.pair_likelihood(T,X,u_BT,v)
print(likelihood_BT)

In [ ]:
BT_top10 = np.argsort(u_BT)[-20:][::-1]
u_t10_BT = u_plusDC[BT_top10]

for i,index in enumerate(BT_top10):
    player_name = [key for key, value in playerID.items() if value == index][0]
    print(f'top-{i+1}: player name:{player_name}, score:{u_t10_BT[i]}')

### Aging effect

Basis functions

In [ ]:
[parameters[ss] for ss in s]

In [ ]:
x = np.linspace(8,64,100)

basis_functions_set = [gauss_kernel(x,parameters[ss][0],parameters[ss][1]) for ss in s]
text = ['$(25, 0.01)$', '$(25, 0.03)$', '$(30, 0.01)$', '$(30, 0.03)$', '$(35, 0.01)$', '$(35, 0.03)$']
color = ['slateblue','slateblue','coral','coral','deeppink','deeppink']
linestyle = ['-',':','-',':','-',':']

fig,ax = plt.subplots(figsize=(11.3,10))
for ss in s:
    ax.plot(x, gauss_kernel(x,parameters[ss][0],parameters[ss][1]), 
    color = color[ss], linewidth = 3,linestyle=linestyle[ss], label = text[ss])

sz = 36
ax.set_xlabel('Age', size = sz)
ax.set_ylabel('Value', size = sz)
ax.set_ylim([-0.05, 1.18])
plt.xticks(size = sz)
plt.yticks(size = sz)

rc('text', usetex=True)
rc('font', family='serif')

ax.legend(title='Selected $(a, \lambda)$', prop={'size': sz},\
         title_fontsize=sz, loc='upper right')

plt.grid()

plt.savefig(image_path+'basis.pdf')
plt.show()

Aging

In [ ]:
age_effect = lambda t: sum([v_plusDC[i]*gauss_kernel(t,parameters[ss][0],parameters[ss][1]) for i,ss in enumerate(s)])

In [ ]:
v_plusDC

In [ ]:
#plot
x = np.linspace(8,64,100)
y = age_effect(x)
fig,ax = plt.subplots(figsize=(11.3,10))
ax.plot(x, y, color = 'black', linewidth = 5)

ax.axvline(x=17.4, color='red', linestyle=':', linewidth = 3)
ax.axvline(x=36.6, color='red', linestyle=':', linewidth = 3)
ax.fill_betweenx(y*100-20, 17.4, 36.6, color='gray', alpha=0.12)

sz = 36
ax.set_xlabel('Age', size = sz)
ax.set_ylabel('Aging effect', size = sz)
ax.set_ylim([-0.2, 4.4])
plt.xticks(size = sz)
plt.yticks(size = sz)

rc('text', usetex=True)
rc('font', family='serif')
plt.grid()
plt.savefig(image_path+'age_effect.pdf')
plt.show()

Dynamic score

In [ ]:
# setting
total_num = 10
players = top_player[:total_num]
players_information = {}

In [ ]:
with open(data_path+'birthday.txt','r') as f:
    lines = f.readlines()[:total_num]
for line,player in zip(lines,players):
    win_age = df[df['winner_name'] == player]['winner_age']
    lose_age = df[df['loser_name'] == player]['loser_age']

    birthday = float(line[:-1].split(':')[1])
    start_year = birthday+min(min(win_age),min(lose_age),19)
    end_year = birthday+max([max(win_age),max(lose_age)])

    temp = {'birthday':birthday,
            'start_year':start_year,
            'end_year':end_year,
            }

    players_information[player] = temp
    

In [ ]:
fig,ax = plt.subplots(figsize=(90,30))
sz = 105
ax.set_xlabel('Year', size = sz)
ax.set_ylabel('Log score', size = sz)
ax.set_ylim([3.3, 8.1])
ax.set_xlim([1969, 2025])
plt.xticks(size = sz)
plt.yticks(size = sz)
rc('text', usetex=True)
rc('font', family='serif')
year_tag = [('goldenrod', '--'),('red', '--'), ('salmon', '--'),\
            ('royalblue', '--'), ('limegreen', '--'), ('cyan', '--'),\
            ('magenta', '--'), ('springgreen', '--'), ('indianred', '--'), \
            ('darkcyan', '--')]
year_tag = [('goldenrod', '--'),('red', '--'), ('salmon', '--'),\
            ('royalblue', '--'), ('limegreen', '--'), ('cyan', '--'),\
            ('magenta', '--'), ('springgreen', '--'), ('indianred', '--'), \
            ('darkcyan', '--')]


for i,player in enumerate(players):

    start_year = players_information[player]['start_year']
    end_year = players_information[player]['end_year']
    birthday = players_information[player]['birthday']

    period = np.linspace(start_year, end_year, 200)
    
    u = u_plusDC[playerID[player]]
    score = age_effect(period-birthday)+u

    
    ax.plot(period,score,color = year_tag[i][0], 
            linestyle = year_tag[i][1], linewidth = 6, label = top_player[i])
ax.legend(prop={'size': 60}, title_fontsize=sz, loc = 'lower left', bbox_to_anchor=(0, 0))


plt.grid()
plt.savefig(image_path+'dynamic_score.pdf')
plt.show()

### Does covariate fully capture the score?

In [ ]:
from sklearn.cluster import KMeans
np.random.seed(100)
Num = [4,6,8,10,12,14]
XX = np.array([x[0,:]-x[1,:] for x in X])
def minimum_cluster(XX,num):
    v_hat = algorithm.estimate_v(XX,d)
    p_hat = algorithm.s(XX@v_hat)
    kmeans = KMeans(n_clusters=num)
    kmeans.fit(p_hat.reshape(-1,1))
    labels = kmeans.labels_
    labels_num = [sum(labels==i) for i in range(num)]
    print(f'The minimum length of clusters: {min(labels_num)}')


In [ ]:
PP = []
for num in Num:
    chi,p_value = algorithm.test_statistics(XX,d,num)
    PP.append(p_value)
    print(f"num:{num},chi2: {chi} and p value: {p_value}")


#### Simulate comparisons

In [ ]:
v_hat = algorithm.estimate_v(XX,d)
repeat_times = 10000
tasks = [delayed(algorithm.test_statistics)(XX,d,num,v_hat) for _ in range(repeat_times)]
results = Parallel(n_jobs=3)(tasks)
results = np.array(results)
with open(data_path+f'goodness-of-fit({num}).txt','a') as file:
    for r in results:
        for rr in r:
            file.write(f'{str(rr)},')
        file.write('\n')

In [ ]:
statistics,p_value = [],[]
with open(data_path+f'goodness-of-fit({num}).txt','r') as file:
    for line in file:
        temp = line.split(',')
        statistics.append(float(temp[0]))
        p_value.append(float(temp[1]))

In [ ]:
from scipy.stats import chi2
plt.figure(figsize=(10.8, 10))
x = np.linspace(0,3*num, 1000)
pdf = chi2.pdf(x, num-1)
plt.hist(statistics, bins=100, alpha=0.7, color='yellow', edgecolor='black',density = True)
plt.plot(x, pdf, label=fr'$\chi^2({num-1})$ Distribution', color='blue')
plt.title(fr"Distribution of statistic ($K={num}$)",size =sz)
plt.xlabel('Statistic',size =sz)
plt.ylabel('Probability density',size =sz)
plt.xticks(size=sz)
plt.yticks(size=sz)
plt.legend(fontsize=sz/2)
plt.savefig(image_path+f'statistics({num}).pdf')


In [ ]:
plt.figure(figsize=(10.8, 10))
plt.hist(p_value, bins=10, alpha=0.7, color='skyblue', edgecolor='black',density = True)
plt.title(fr"Distribution of $p$ value ($K={num}$)",size =sz)
plt.xlabel(r'$p$ value',size=sz)
plt.ylabel('Probability density',size=sz)
plt.xticks(size=sz)
plt.yticks(size=sz)
plt.savefig(image_path+f'p_value({num}).pdf')


### $k$-folder Cross Validation

136-folder CV

In [ ]:
folder_len = 1000
folders = []
np.random.seed(42)
temp_folder = set(range(N))
I = int(N/folder_len)
for i in range(I):
    temp_folder = list(temp_folder)
    index = np.random.choice(temp_folder, folder_len,replace=False)
    index = np.sort(index)[::-1]
    temp_folder = set(temp_folder) - set(index)
    folders.append(index.tolist())
folders[-1] = folders[-1] +list(temp_folder)

#### Loss function

In [ ]:
CV_name = {'Cross entropy':"cross_entropy.txt",
           'AUC':"AUC.txt",
           'Hinge loss':"hinge_loss.txt",
           }

In [ ]:
results = Parallel(n_jobs=10)(delayed(algorithm.tennis_cross_validation)
                              (T,X,n,folder,data_path,CV_name) for folder in folders)

Visualization

In [ ]:
Data = {}
for key in CV_name:
    CV_filename = CV_name[key]
    D = []
    with open(data_path+CV_filename,'r') as file:
        for line in file:
            tem = np.zeros((2))
            for i,num in enumerate(line.split(';')):
                tem[i] = float(num)
            D.append(tem)
    D = np.array(D)
    D = D[~np.isnan(D).any(axis=1)]
    Data[key] = D
Data['Cross entropy'] = -Data['Cross entropy']

In [ ]:
"""legends = [
        r'$\text{Cross entropy}_\text{BT}-\text{Cross entropy}_\text{PlusDC}$',
        r'$\text{AUC}_\text{PlusDC}-\text{AUC}_\text{BT}$',
        r'$\text{Hinge loss}_\text{BT}-\text{Hinge loss}_\text{PlusDC}$',
        ]"""
legends = [
        r'$\rm{CE}_{\rm{BT}}-\rm{CE}_{\rm{PlusDC}}$',
        r'$\rm{AUC}_{\rm{PlusDC}}-\rm{AUC}_{\rm{BT}}$',
        r'$\rm{HL}_{\rm{BT}}-\rm{HL}_{\rm{PlusDC}}$',
        ]
D = []
labels = []
for key in Data:
    Loss = Data[key]
    if key == 'AUC':
        H = Loss[:,1]-Loss[:,0]
    else:
        H = Loss[:,0]-Loss[:,1]
    D.append(H)
    labels.append(key)
D = np.array(D).T
fig, ax = plt.subplots(figsize=(10.8, 10))
colors = ['lightblue', 'orange', 'royalblue']
bx = ax.boxplot(D, tick_labels=labels, patch_artist=True,medianprops={'linewidth': 2})
ax.set_position([0.165, 0.125, 0.8, 0.8])
for median in bx['medians']:
        median.set_color('black')
for patch, color in zip(bx['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_edgecolor('black')
ax.tick_params(axis='y', which='both')  

legend_handles = [plt.Line2D([0], [0], color=color, lw=4, label=legend) 
                  for color,legend in zip(colors,legends)]

# 添加图例
plt.axhline(0, color='grey', linewidth=1, linestyle='--') 
plt.xticks(size=sz/4)
plt.yticks([-.005*(1-i) for i in range(6)],size=sz/2)
plt.savefig(image_path+f'Difference_metrics.pdf')
plt.show()

In [ ]:
for key in Data:
    save_name = key
    labels = ['BT','PlusDC']
    colors = ['lightblue', 'orange']


    fig, ax = plt.subplots(figsize=(10.8, 10))
    bx = ax.boxplot(Data[key], tick_labels=labels, patch_artist=True,medianprops={'linewidth': 2})
    ax.set_position([0.165, 0.125, 0.8, 0.8])
    plt.title(key,size =sz/2)
    plt.xticks(size=sz/4)
    plt.yticks(size=sz/2)
    ax.tick_params(axis='y', which='both')


    for median in bx['medians']:
        median.set_color('black')
    for patch, color in zip(bx['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_edgecolor('black')

    plt.savefig(image_path+f'KFCV_{save_name}.pdf')
    plt.show()

#### AUC

In [ ]:
from sklearn.metrics import roc_curve, auc

def pair_compute_p(T,X,u,v):
    p = []
    for i, t in enumerate(T):
        R = np.exp(u[t] + X[i] @ v)
        tem = R[0] / sum(R)
        p.append(tem)
    p = np.array(p)
    y = np.ones(len(p))
    p = np.concatenate((p,1-p))
    ytrue = np.concatenate((y,1-y))
    fpr, tpr, thresholds = roc_curve(ytrue, p)
    return fpr, tpr

def roc(subset):
    N = len(T)
    Ttrain = [T[i] for i in range(N) if i not in subset]
    Xtrain = [cov[i] for i in range(N) if i not in subset]
    Ttest = [T[i] for i in range(N) if i in subset]
    Xtest = [cov[i] for i in range(N) if i in subset]
    u_pl,v_pl = algorithm.AM(Ttrain,Xtrain,n,P=True,Eu=1e-5,type = 'pair')
    if np.isnan(u_pl).any():
        pass
    else:
        u_plusDC,v_plusDC = algorithm.AM(Ttrain,Xtrain,n,
                                    E=1e-5,Eu=1e-5,Ev=1e-12,
                                    I=52,type = 'pair')
    fpr_plusDC,tpr_plusDC = pair_compute_p(Ttest,Xtest,u_plusDC,v_plusDC)
    fpr_pl,tpr_pl = pair_compute_p(Ttest,Xtest,u_pl,v_pl)
    roc_auc_pl = auc(fpr_pl, tpr_pl)
    roc_auc_plusDC = auc(fpr_plusDC, tpr_plusDC)
    return [[fpr_pl,tpr_pl],[fpr_plusDC,tpr_plusDC]]
def plot_auc(results):
    fpr_pl, tpr_pl = results[0]
    fpr_plusDC, tpr_plusDC = results[1]
    roc_auc_pl = auc(fpr_pl, tpr_pl)
    roc_auc_plusDC = auc(fpr_plusDC, tpr_plusDC)
    #plt.plot(fpr_pl, tpr_pl, color='blue', label='ROC curve (AUC = {:.2f})'.format(roc_auc_pl))
    plt.plot(fpr_pl, tpr_pl, color='blue', label='BT-ROC curve (AUC = {:.2f})'.format(roc_auc_pl))
    #plt.plot(fpr_pl, tpr_pl, color='blue', label='ROC curve (BT)')
    #plt.plot(fpr_plusDC, tpr_plusDC, color='orange', label='ROC curve (AUC = {:.2f})'.format(roc_auc_plusDC))
    plt.plot(fpr_plusDC, tpr_plusDC, color='orange', label='PlusDC-ROC curve (AUC = {:.2f})'.format(roc_auc_plusDC))
    #plt.plot(fpr_plusDC, tpr_plusDC, color='orange', label='ROC curve (PlusDC)')
    plt.plot([0, 1], [0, 1], color='red', linestyle='--')  # 随机猜测的对角线
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic')
    plt.legend(loc='lower right')
    plt.grid()
    plt.show()

In [ ]:
len(T)

In [ ]:
subset = list(range(int(0.001*len(T))))
results= roc(subset)

In [ ]:
int(0.001*len(T))

In [ ]:
Data = {}
for key in CV_name:
    CV_filename = CV_name[key]
    D = []
    with open(data_path+CV_filename,'r') as file:
        for line in file:
            tem = np.zeros((2))
            for i,num in enumerate(line.split(';')):
                tem[i] = float(num)
            D.append(tem)
    D = np.array(D)
    D = D[~np.isnan(D).any(axis=1)]
    Data[key] = D
Data['Cross entropy'] = -Data['Cross entropy']

In [ ]:
plot_auc(results)